In [ ]:
# Ячейка 1
import pandas as pd
import numpy as np
import sys
sys.path.append('..')
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import lab_utils

cardio = pd.read_csv('../01-feature-importance-and-selection/data/cardiovascular_risk.csv')
credit = pd.read_csv('../01-feature-importance-and-selection/data/credit_risk.csv')
credit['loan_purpose'] = LabelEncoder().fit_transform(credit['loan_purpose'])

artifacts = lab_utils.load_upstream_artifacts()
features = artifacts['feature_sets']['cardiovascular_risk']['set_A']
X = cardio[features]
y = cardio['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000).fit(X_train_s, y_train)
y_pred = model.predict(X_test_s)
y_prob = model.predict_proba(X_test_s)[:,1]

print(f'Ошибок: {(y_pred != y_test).sum()} из {len(y_test)}')
print(f'FP: {((y_pred==1)&(y_test==0)).sum()}, FN: {((y_pred==0)&(y_test==1)).sum()}')

In [ ]:
# Ячейка 2: Анализ ошибок
errors_idx = np.where(y_pred != y_test)[0][:5]
explanations = []

for idx in errors_idx:
    orig = X_test_s[idx].copy()
    error_type = 'FP' if (y_test.iloc[idx]==0 and y_pred[idx]==1) else 'FN'
    
    for i, f in enumerate(features):
        pert = orig.copy()
        pert[i] = X_train_s[:,i].mean()
        new_prob = model.predict_proba([pert])[0,1]
        imp = abs(y_prob[idx] - new_prob)
        
        explanations.append({
            'dataset':'cardio','model':'LR','feature_set':'set_A',
            'case_group_index':idx,'error_type':error_type,
            'y_true':y_test.iloc[idx],'y_pred':y_pred[idx],
            'score':y_prob[idx],'score_source':'predict_proba',
            'explanation_method':'perturbation','feature':f,
            'importance_value':imp,'detail_a':orig[i],'detail_b':pert[i]
        })

error_df = pd.DataFrame(explanations)
error_df.to_csv('outputs/error_case_explanations.csv', index=False)
print(error_df[['error_type','feature','importance_value']])

In [ ]:
# Ячейка 3: Сегменты
X_test_df = X_test.copy()
X_test_df['segment'] = pd.cut(X_test_df['age'], bins=[0,45,60,100], labels=['young','middle','senior'])
X_test_df['error'] = y_pred != y_test.values
print(X_test_df.groupby('segment')['error'].mean())